# 🛡️ Engineering Track: Autonomous Vulnerability Discovery & Verification Engine
### Systems Architecture: Compiler AST Mechanics, Type-Safe Pydantic Contracts, and Dynamic Sandbox Testing

Welcome to the **Engineering Track** of the Cyber Test Automation curriculum. This track is designed for Computer Science undergraduates, AppSec engineers, and CTF competitors seeking a systems-level understanding of vulnerability discovery, automated remediation, and closed-loop verification using Google Gemini reasoning models.

---

## Systems Foundation: The Parser Boundary & AST Mechanics

At the hardware and systems level, code injection vulnerabilities stem from the fundamental **Von Neumann architecture** characteristic: instructions and data share the same representation and execution stream.

### How SQLite Compiles & Executes a Query
When an application queries SQLite, the database engine processes the SQL statement through a four-phase compiler pipeline:

```text
Raw SQL String ──> [ 1. Tokenizer / Lexer ] ──> Token Stream
                                                      │
                                                      ▼
                                            [ 2. Lemon Parser ]
                                                      │
                                                      ▼
                                          Abstract Syntax Tree (AST)
                                                      │
                                                      ▼
                                            [ 3. Code Generator ]
                                                      │
                                                      ▼
                                             VDBE Bytecode
                                                      │
                                                      ▼
                                            [ 4. VDBE Engine ] ──> B-Tree Storage Engine
```

### Insecure String Interpolation vs. Parameter Binding
- **The Flaw (String Interpolation):** When user input is directly concatenated (`f"... WHERE username = '{username}'"`), the untrusted payload enters at **Phase 1 (Tokenizer)**. Characters such as `'`, `OR`, and `--` are tokenized as grammar keywords and operators, fundamentally mutating the **Abstract Syntax Tree (AST)**.
- **The Mitigation (Parameter Binding / Prepared Statements):** The SQL statement structure is parsed and compiled with placeholder tokens (`OP_Variable` in VDBE). Untrusted inputs are bound directly into virtual machine registers at **Phase 4 (VDBE Runtime)** as scalar literal data. The tokenizer and parser are never re-invoked on the user data, mathematically preventing AST boundary violations.

### Gemini Cyber Reasoning Benchmarks
| Benchmark / Capability | Gemini 3.8 Flash Cyber | Industry Standard |
| :--- | :--- | :--- |
| **CyberGym Overall Score** | **86.2%** | ~83.8% (Anthropic Opus 5) |
| **Vulnerability Discovery** | **>70%** (across 20 languages) | Highly variable |
| **CWE-Bench Patching Rate** | **47.2%** | Limited autonomous remediation |
| **Chrome Security Patching** | **2.6x more correct patches** | Standard LLMs |

---

## Environment Setup
```bash
# Synchronize project dependencies using uv
uv sync

# Configure Gemini API credentials in project root .env
cp .env.example .env

# Launch Jupyter
uv run jupyter notebook
```


## 1. Environment Initialization & Dynamic Model Selector

We initialize the Google GenAI SDK client (`genai.Client()`) with automatic credential discovery via `python-dotenv`. The dynamic selector defaults to **`gemini-3.8-flash`** for cutting-edge cyber reasoning speed and accuracy, with **`gemini-2.5-flash`** and **`gemini-2.5-pro`** available as enterprise fallbacks.

In [1]:
import os
import sys
import ast
import difflib
import sqlite3
import ipywidgets as widgets
from dotenv import load_dotenv, find_dotenv
from google import genai
from google.genai import types
from google.genai.errors import ServerError, ClientError
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from pydantic import BaseModel, Field
from IPython.display import display, Markdown, HTML, clear_output

# Load environment variables searching up to repository root
load_dotenv(find_dotenv())
client = genai.Client()

# Dynamic Model Selector
AVAILABLE_MODELS = [
    ("Gemini 3.8 Flash (Default - High Speed & Cyber Reasoning)", "gemini-3.8-flash"),
    ("Gemini 2.5 Flash (High Availability Production Fallback)", "gemini-2.5-flash"),
    ("Gemini 2.5 Pro (Deep Multi-Step Reasoning)", "gemini-2.5-pro"),
]

model_dropdown = widgets.Dropdown(
    options=AVAILABLE_MODELS,
    value="gemini-3.8-flash",
    description="Target Model:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="620px")
)

display(Markdown("### ⚙️ Client Initialized & Dynamic Model Selector"))
display(Markdown("Select the target model for the vulnerability audit (defaults to **`gemini-3.8-flash`**):"))
display(model_dropdown)


### ⚙️ Client Initialized & Dynamic Model Selector

Select the target model for the vulnerability audit (defaults to **`gemini-3.8-flash`**):

Dropdown(description='Target Model:', layout=Layout(width='620px'), options=(('Gemini 3.8 Flash (Default - Hig…

## 2. Defining Target Attack Surface & In-Memory Sandbox

Below is a target authentication function `authenticate_user(username, password_hash)`. Notice line 7 where direct string interpolation (`f"...{username}..."`) allows an external attacker to alter the query's AST logic.

In [2]:
# Vulnerable Target Code containing CWE-89
vulnerable_code = """import sqlite3

def authenticate_user(username: str, password_hash: str):
    conn = sqlite3.connect('auth_sandbox.db')
    cursor = conn.cursor()
    # VULNERABILITY (CWE-89): Direct string interpolation bypasses AST boundaries
    query = f"SELECT id, username, role FROM users WHERE username = '{username}' AND password_hash = '{password_hash}'"
    cursor.execute(query)
    record = cursor.fetchone()
    conn.close()
    return record
"""

def setup_sandbox_database(db_path="auth_sandbox.db"):
    """Initializes an isolated SQLite database with realistic user credentials and role claims."""
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("DROP TABLE IF EXISTS users")
    cur.execute("CREATE TABLE users (id INTEGER PRIMARY KEY, username TEXT, password_hash TEXT, role TEXT)")
    cur.executemany("INSERT INTO users VALUES (?, ?, ?, ?)", [
        (1, 'alice', 'sha256$5e884898da28047151d0e56f8dc6292773603d0d6aabbdd62a11ef721d1542d8', 'engineer'),
        (2, 'bob', 'sha256$4e07408562bedb8b60ce05c1decfe3ad16b72230967de01f640b7e4729b49fce', 'analyst'),
        (3, 'secops_admin', 'sha256$2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824', 'superadmin'),
    ])
    conn.commit()
    conn.close()

setup_sandbox_database()
display(Markdown("### 🎯 Target Code & Ephemeral Sandbox Database Configured"))
display(Markdown(f"```python\n{vulnerable_code}\n```"))


### 🎯 Target Code & Ephemeral Sandbox Database Configured

```python
import sqlite3

def authenticate_user(username: str, password_hash: str):
    conn = sqlite3.connect('auth_sandbox.db')
    cursor = conn.cursor()
    # VULNERABILITY (CWE-89): Direct string interpolation bypasses AST boundaries
    query = f"SELECT id, username, role FROM users WHERE username = '{username}' AND password_hash = '{password_hash}'"
    cursor.execute(query)
    record = cursor.fetchone()
    conn.close()
    return record

```

### 2.1 Interactive CTF Triage & Exploitation Console

In offensive security and CTF competitions, SQL injection is triaged across distinct attack patterns:
1. **Tautology / Auth Bypass (`' OR '1'='1`):** Forces the `WHERE` clause condition to evaluate to boolean `TRUE` regardless of credential validity.
2. **Line Comment / Privilege Hijack (`secops_admin' --`):** Truncates the SQL statement after the injected username, ignoring the password check entirely.
3. **UNION-Based Schema Extraction (`' UNION SELECT 999, 'injected', 'superadmin' --`):** Leverages SQLite's set operators to construct arbitrary synthetic records.
4. **Boolean Blind Logic Probe (`' OR (SELECT 1)=1 --`):** Validates parser execution pathways when application errors are suppressed.

Use the console below to execute each payload against the live SQLite VDBE engine.

In [3]:
ctf_payloads = [
    ("Vector 1: Auth Bypass via Tautology (' OR '1'='1)", "' OR '1'='1", "dummy_hash"),
    ("Vector 2: Privilege Hijack via Comment Truncation (secops_admin' --)", "secops_admin' --", ""),
    ("Vector 3: UNION-Based Synthetic Record Injection", "' UNION SELECT 999, 'attacker_root', 'superadmin' --", ""),
    ("Vector 4: Legitimate Authentication (alice)", "alice", "sha256$5e884898da28047151d0e56f8dc6292773603d0d6aabbdd62a11ef721d1542d8"),
]

ctf_selector = widgets.RadioButtons(
    options=[p[0] for p in ctf_payloads],
    description="Attack Vector:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="680px")
)

custom_user_input = widgets.Text(
    placeholder="Or input custom payload... (e.g., admin' /*)",
    description="Custom Payload:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="550px")
)

ctf_btn = widgets.Button(
    description="Execute Exploitation Probe",
    button_style="danger",
    icon="terminal",
    layout=widgets.Layout(width="240px")
)

ctf_output = widgets.Output()

def on_ctf_probe_clicked(b):
    with ctf_output:
        clear_output()
        if custom_user_input.value.strip():
            user_val = custom_user_input.value.strip()
            pass_val = ""
        else:
            choice = ctf_selector.value
            entry = next(p for p in ctf_payloads if p[0] == choice)
            user_val, pass_val = entry[1], entry[2]
            
        conn = sqlite3.connect('auth_sandbox.db')
        cur = conn.cursor()
        raw_query = f"SELECT id, username, role FROM users WHERE username = '{user_val}' AND password_hash = '{pass_val}'"
        
        display(Markdown(f"**Generated SQL Query:**\n```sql\n{raw_query}\n```"))
        try:
            res = cur.execute(raw_query).fetchall()
            conn.close()
            display(Markdown(f"**Database VDBE Result:** `{res}`"))
            
            if any('admin' in str(r).lower() for r in res) and user_val != 'alice':
                display(Markdown("🚨 **CRITICAL EXPLOIT SUCCESS (Privilege Compromise):**\n"
                                 "The parser executed the injected tokens, authenticating as `superadmin` or dumping arbitrary records!"))
            elif len(res) > 0 and user_val == 'alice':
                display(Markdown("✅ **Legitimate Authentication:** Authentic credentials verified successfully."))
            elif len(res) > 0:
                display(Markdown(f"⚠️ **Authentication Bypass:** Evaluated truthy, returning {len(res)} records."))
            else:
                display(Markdown("🛡️ **Access Denied:** Query evaluated falsy; no records returned."))
        except Exception as err:
            display(Markdown(f"💥 **Parser / VDBE Runtime Exception:** `{err}`"))

ctf_btn.on_click(on_ctf_probe_clicked)

display(Markdown("### 🎛️ Interactive CTF Triage Console"))
display(ctf_selector)
display(custom_user_input)
display(ctf_btn)
display(ctf_output)


### 🎛️ Interactive CTF Triage Console

RadioButtons(description='Attack Vector:', layout=Layout(width='680px'), options=("Vector 1: Auth Bypass via T…

Text(value='', description='Custom Payload:', layout=Layout(width='550px'), placeholder="Or input custom paylo…

Button(button_style='danger', description='Execute Exploitation Probe', icon='terminal', layout=Layout(width='…

Output()

## 3. Type-Safe Contract Enforcement via Pydantic v2

In production DevSecOps and SAST/DAST automation, unstructured natural language responses cannot be reliably integrated into CI/CD quality gates.

We define a strict **Pydantic schema** (`DevSecOpsRemediationReport`) incorporating formal **MITRE CWE** taxonomies, **CVSS v3.1** vector metrics, systems-level root-cause analysis, and atomic patched source code.

In [4]:
class DevSecOpsRemediationReport(BaseModel):
    cwe_id: str = Field(description="The formal CWE identifier, e.g. 'CWE-89'")
    vulnerability_class: str = Field(description="Standard vulnerability category name, e.g. 'SQL Injection'")
    cvss_score: float = Field(description="CVSS v3.1 Base Score between 0.0 and 10.0", ge=0.0, le=10.0)
    cvss_vector: str = Field(description="Standard CVSS v3.1 vector string, e.g. 'CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H'")
    root_cause_analysis: str = Field(description="Systems-level explanation of the parser-level boundary violation.")
    vulnerable_lines: list[int] = Field(description="1-indexed line numbers in the original snippet containing the flaw.")
    remediated_code: str = Field(description="Hardened, syntactically valid Python code using parameterized query bindings and context managers.")
    patch_strategy: str = Field(description="Technical rationale for the applied architectural mitigation.")

display(Markdown("✅ **Pydantic Contract `DevSecOpsRemediationReport` declared.**"))


✅ **Pydantic Contract `DevSecOpsRemediationReport` declared.**

## 4. Single-Shot LLM Audit & Autonomous Remediation Execution

We invoke the Gemini API using `response_mime_type="application/json"` and `response_schema=DevSecOpsRemediationReport`. The call is wrapped in exponential backoff via `tenacity` to ensure resilience against enterprise rate limits.

In [5]:
selected_model = model_dropdown.value if "model_dropdown" in globals() else "gemini-3.8-flash"
display(Markdown(f"*Executing autonomous security audit using:* **`{selected_model}`**..."))

appsec_prompt = f"""You are a Principal Application Security Engineer auditing code in an automated DevSecOps CI/CD pipeline.
Analyze the following Python snippet for security vulnerabilities.
Determine the exact CWE, calculate the CVSS v3.1 Base Score and Vector string, explain the parser-level root cause, and synthesize a production-grade, hardened patch using parameterized queries and context managers.

Target Code:
{vulnerable_code}
"""

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=6),
    retry=retry_if_exception_type(ServerError),
    before_sleep=lambda r: display(Markdown(f"⚠️ *API busy (503/500). Retrying in {r.next_action.sleep:.1f}s...*"))
)
def _raw_audit_call(model_name: str, code_prompt: str) -> types.GenerateContentResponse:
    return client.models.generate_content(
        model=model_name,
        contents=code_prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=DevSecOpsRemediationReport,
            temperature=0.1,  # Low temperature for deterministic engineering output
        )
    )

def run_vulnerability_audit(model_name: str, code_prompt: str) -> types.GenerateContentResponse:
    try:
        return _raw_audit_call(model_name, code_prompt)
    except Exception as err:
        fallback = "gemini-2.5-flash"
        if model_name != fallback:
            display(Markdown(f"⚠️ *Model `{model_name}` temporarily unavailable. Automatically failing over to resilient `{fallback}`...*"))
            return _raw_audit_call(fallback, code_prompt)
        raise

def render_remediation_report(rep: DevSecOpsRemediationReport):
    severity_color = "#d32f2f" if rep.cvss_score >= 9.0 else "#e65100" if rep.cvss_score >= 7.0 else "#f57c00"
    cwe_num = rep.cwe_id.replace("CWE-", "").strip()
    
    report_md = f"""
### 🛡️ Enterprise DevSecOps Audit Findings

| Metric | Specification |
| :--- | :--- |
| **Vulnerability Class** | **{rep.vulnerability_class}** |
| **MITRE Taxonomy** | [`{rep.cwe_id}`](https://cwe.mitre.org/data/definitions/{cwe_num}.html) |
| **CVSS v3.1 Base Score** | <span style="color:{severity_color}; font-weight:bold; font-size:1.15em;">{rep.cvss_score} / 10.0</span> |
| **CVSS Vector** | `{rep.cvss_vector}` |
| **Vulnerable Lines** | `{rep.vulnerable_lines}` |

#### 🔬 Parser Root-Cause Analysis
> {rep.root_cause_analysis}

#### 🔧 Hardened Remediated Code
```python
{rep.remediated_code.strip()}
```

#### 📋 Architectural Patch Strategy
{rep.patch_strategy}
"""
    display(Markdown(report_md))

try:
    response = run_vulnerability_audit(selected_model, appsec_prompt)
    report = DevSecOpsRemediationReport.model_validate_json(response.text)
    render_remediation_report(report)
except Exception as e:
    display(Markdown(f"❌ **Unexpected Error:** `{e}`"))


*Executing autonomous security audit using:* **`gemini-3.8-flash`**...

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


⚠️ *API busy (503/500). Retrying in 2.0s...*


### 🛡️ Enterprise DevSecOps Audit Findings

| Metric | Specification |
| :--- | :--- |
| **Vulnerability Class** | **SQL Injection** |
| **MITRE Taxonomy** | [`CWE-89`](https://cwe.mitre.org/data/definitions/89.html) |
| **CVSS v3.1 Base Score** | <span style="color:#d32f2f; font-weight:bold; font-size:1.15em;">9.8 / 10.0</span> |
| **CVSS Vector** | `CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H` |
| **Vulnerable Lines** | `[7, 8]` |

#### 🔬 Parser Root-Cause Analysis
> The function performs dynamic SQL construction via Python f-string formatting, concatenating untrusted user input directly into the command string prior to lexical analysis and parsing by the SQLite engine. Because user inputs are not separated from query syntax, an attacker can supply SQL metacharacters (such as single quotes and boolean operators) that alter the Abstract Syntax Tree (AST) generated by SQLite's SQL parser, enabling authentication bypass and unauthorized data extraction.

#### 🔧 Hardened Remediated Code
```python
import sqlite3
from typing import Optional, Tuple, Any

def authenticate_user(username: str, password_hash: str) -> Optional[Tuple[Any, ...]]:
    query = "SELECT id, username, role FROM users WHERE username = ? AND password_hash = ?"
    with sqlite3.connect('auth_sandbox.db') as conn:
        cursor = conn.cursor()
        cursor.execute(query, (username, password_hash))
        return cursor.fetchone()
```

#### 📋 Architectural Patch Strategy
Separate SQL code structure from untrusted data by using parameterized statements with positional placeholders (?), ensuring the SQLite parser treats user arguments strictly as literal values rather than executable syntax. Wrap database connection handling in context managers ('with' statements) to guarantee deterministic resource management and lifecycle cleanup.


## 5. Syntactic Diff Inspection & AST Validation

Before deploying code synthesized by an LLM to a repository, automated CI/CD pipelines must enforce two quality gates:
1. **Abstract Syntax Tree (AST) Validation:** Compile the patch with Python's built-in `ast.parse()` to guarantee syntax correctness before execution.
2. **Line-Level Unified Diff Generation:** Inspect code modifications via `difflib.unified_diff` to confirm that changes adhere strictly to the principle of minimal intervention.

In [6]:
def validate_ast_syntax(code_str: str) -> tuple[bool, str]:
    """Validates that a code string compiles into a valid Python Abstract Syntax Tree."""
    try:
        ast.parse(code_str)
        return True, "AST validation successful (zero syntax errors)"
    except SyntaxError as e:
        return False, f"SyntaxError on line {e.lineno}: {e.msg}"

def display_patch_diff(original: str, patched: str):
    """Renders a unified diff between insecure and hardened code."""
    orig_lines = original.strip().splitlines()
    patch_lines = patched.strip().splitlines()
    diff = list(difflib.unified_diff(
        orig_lines,
        patch_lines,
        fromfile="insecure_baseline.py (AST Injection Vector)",
        tofile="hardened_patch.py (Parameterized Prepared Statement)",
        lineterm=""
    ))
    if diff:
        display(Markdown(f"### 🔍 Syntactic Unified Diff\n```diff\n{'\n'.join(diff)}\n```"))
    else:
        display(Markdown("⚠️ *No code differences detected.*\n"))

if 'report' in globals():
    valid, ast_msg = validate_ast_syntax(report.remediated_code)
    display(Markdown(f"**AST Compilation Status:** `{'✅ ' + ast_msg if valid else '❌ ' + ast_msg}`"))
    display_patch_diff(vulnerable_code, report.remediated_code)
else:
    display(Markdown("⚠️ *Run Cell 4 to generate the audit report before running diff analysis.*\n"))


**AST Compilation Status:** `✅ AST validation successful (zero syntax errors)`

### 🔍 Syntactic Unified Diff
```diff
--- insecure_baseline.py (AST Injection Vector)
+++ hardened_patch.py (Parameterized Prepared Statement)
@@ -1,11 +1,9 @@
 import sqlite3
+from typing import Optional, Tuple, Any
 
-def authenticate_user(username: str, password_hash: str):
-    conn = sqlite3.connect('auth_sandbox.db')
-    cursor = conn.cursor()
-    # VULNERABILITY (CWE-89): Direct string interpolation bypasses AST boundaries
-    query = f"SELECT id, username, role FROM users WHERE username = '{username}' AND password_hash = '{password_hash}'"
-    cursor.execute(query)
-    record = cursor.fetchone()
-    conn.close()
-    return record
+def authenticate_user(username: str, password_hash: str) -> Optional[Tuple[Any, ...]]:
+    query = "SELECT id, username, role FROM users WHERE username = ? AND password_hash = ?"
+    with sqlite3.connect('auth_sandbox.db') as conn:
+        cursor = conn.cursor()
+        cursor.execute(query, (username, password_hash))
+        return cursor.fetchone()
```

## 6. Closed-Loop Dynamic Verification Harness (Blast Chamber)

In mission-critical security pipelines, an AI-generated patch cannot be accepted based on text generation alone. We execute a **closed-loop dynamic verification harness**:
1. **Baseline Exploit:** Run the exploit payload (`secops_admin' --`) against the unpatched function, asserting authentication bypass.
2. **Dynamic Namespace Isolation:** Compile `remediated_code` inside an isolated Python execution namespace using `exec()`.
3. **Neutralization Assertion:** Re-run the exploit against the remediated function, asserting that bypass fails.
4. **Regression Assertion:** Test legitimate user credentials (`alice`), asserting that authentic user workflows remain functional.

In [7]:
# Ensure clean sandbox database
setup_sandbox_database()

# 1. Baseline Exploit Test
baseline_scope = {"sqlite3": sqlite3}
exec(vulnerable_code, baseline_scope)
exploit_user = "secops_admin' --"
exploit_pass = "irrelevant_payload"

baseline_result = baseline_scope["authenticate_user"](exploit_user, exploit_pass)

# 2. Remediated Patch Test
remediated_scope = {"sqlite3": sqlite3}
if 'report' in globals():
    exec(report.remediated_code, remediated_scope)
    patched_exploit_result = remediated_scope["authenticate_user"](exploit_user, exploit_pass)
    patched_legit_result = remediated_scope["authenticate_user"](
        "alice",
        "sha256$5e884898da28047151d0e56f8dc6292773603d0d6aabbdd62a11ef721d1542d8"
    )
    
    exploit_neutralized = (patched_exploit_result is None)
    regression_cleared = (patched_legit_result is not None and patched_legit_result[1] == "alice")
    
    verification_cert = f"""
### 🧪 Dynamic Verification Certificate

| Test Suite Stage | Target Artifact | Observed Output | Assertion Status |
| :--- | :--- | :--- | :--- |
| **1. Insecure Baseline Exploit** | `vulnerable_code` | Leaked Record: `{baseline_result}` | <span style="color:#d32f2f; font-weight:bold;">VULNERABLE ❌</span> |
| **2. Exploit Neutralization Test** | `remediated_code` | Returned: `{patched_exploit_result}` | <span style="color:#388e3c; font-weight:bold;">{'NEUTRALIZED ✅' if exploit_neutralized else 'BYPASS DETECTED ❌'}</span> |
| **3. Functional Regression Test** | `remediated_code` | Authenticated: `{patched_legit_result}` | <span style="color:#388e3c; font-weight:bold;">{'PASS ✅' if regression_cleared else 'REGRESSION ❌'}</span> |

> **🎯 Autonomous Remediation Gate Verdict:** **{'PRODUCTION READY: VERIFIED & HARDENED ✅' if (exploit_neutralized and regression_cleared) else 'PIPELINE BLOCKED: VERIFICATION FAILED ❌'}**
"""
    display(Markdown(verification_cert))
else:
    display(Markdown("⚠️ *Run Cell 4 to generate the remediation report before testing.*\n"))



### 🧪 Dynamic Verification Certificate

| Test Suite Stage | Target Artifact | Observed Output | Assertion Status |
| :--- | :--- | :--- | :--- |
| **1. Insecure Baseline Exploit** | `vulnerable_code` | Leaked Record: `(3, 'secops_admin', 'superadmin')` | <span style="color:#d32f2f; font-weight:bold;">VULNERABLE ❌</span> |
| **2. Exploit Neutralization Test** | `remediated_code` | Returned: `None` | <span style="color:#388e3c; font-weight:bold;">NEUTRALIZED ✅</span> |
| **3. Functional Regression Test** | `remediated_code` | Authenticated: `(1, 'alice', 'engineer')` | <span style="color:#388e3c; font-weight:bold;">PASS ✅</span> |

> **🎯 Autonomous Remediation Gate Verdict:** **PRODUCTION READY: VERIFIED & HARDENED ✅**


## 7. Multi-Class Vulnerability Suite (Enterprise SAST Benchmark)

Production security audits must encompass a broader taxonomy than SQL injection. Below, we curate an enterprise vulnerability suite covering four critical weaknesses:
1. **CWE-22 (Path Traversal):** Unsanitized `os.path.join` allowing directory traversal via `../../etc/shadow`.
2. **CWE-78 (OS Command Injection):** Use of `shell=True` in `subprocess` allowing execution of chained shell commands (`8.8.8.8; id`).
3. **CWE-918 (Server-Side Request Forgery - SSRF):** Unvalidated external URL fetching allowing extraction of cloud metadata (`169.254.169.254/latest/meta-data/`).
4. **CWE-502 (Insecure Deserialization):** Insecure deserialization via `pickle.loads()` allowing arbitrary RCE through Python `__reduce__` gadgets.

In [8]:
VULNERABILITY_SUITE = {
    "Path Traversal (CWE-22)": """import os

def read_user_profile(filename: str):
    # VULNERABILITY (CWE-22): Direct concatenation allows path traversal (e.g. '../../etc/shadow')
    base_dir = '/var/data/profiles'
    filepath = os.path.join(base_dir, filename)
    with open(filepath, 'r') as f:
        return f.read()
""",
    "OS Command Injection (CWE-78)": """import subprocess

def ping_server(hostname: str):
    # VULNERABILITY (CWE-78): shell=True allows command chaining (e.g. '8.8.8.8; whoami')
    cmd = f"ping -c 1 {hostname}"
    result = subprocess.check_output(cmd, shell=True)
    return result.decode()
""",
    "Server-Side Request Forgery (CWE-918)": """import requests

def fetch_remote_avatar(avatar_url: str):
    # VULNERABILITY (CWE-918): Unrestricted URL fetch allows SSRF against metadata (169.254.169.254)
    response = requests.get(avatar_url, timeout=5)
    return response.content
""",
    "Insecure Deserialization (CWE-502)": """import pickle
import base64

def restore_session(token_b64: str):
    # VULNERABILITY (CWE-502): Unpickling untrusted payload executes arbitrary code via __reduce__
    raw_bytes = base64.b64decode(token_b64)
    return pickle.loads(raw_bytes)
"""
}

suite_selector = widgets.Dropdown(
    options=list(VULNERABILITY_SUITE.keys()),
    value="Path Traversal (CWE-22)",
    description="Select Weakness:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="460px")
)

audit_single_btn = widgets.Button(
    description="Audit Selected Flaw",
    button_style="info",
    icon="search",
    layout=widgets.Layout(width="200px")
)

suite_output = widgets.Output()

def on_audit_single_clicked(b):
    with suite_output:
        clear_output()
        target_name = suite_selector.value
        code_snippet = VULNERABILITY_SUITE[target_name]
        target_model = model_dropdown.value if "model_dropdown" in globals() else "gemini-3.8-flash"
        
        display(Markdown(f"### 🎯 Target Snippet: `{target_name}`"))
        display(Markdown(f"```python\n{code_snippet}\n```"))
        display(Markdown(f"*Auditing with model:* **`{target_model}`**..."))
        
        prompt_text = f"""Analyze the following Python snippet for security weaknesses.
Determine the CWE, CVSS v3.1 score and vector, root cause, and synthesize a production-grade hardened patch.

Code:
{code_snippet}
"""
        try:
            res = run_vulnerability_audit(target_model, prompt_text)
            parsed = DevSecOpsRemediationReport.model_validate_json(res.text)
            render_remediation_report(parsed)
            display_patch_diff(code_snippet, parsed.remediated_code)
        except Exception as err:
            display(Markdown(f"❌ **Audit Exception:** `{err}`"))

audit_single_btn.on_click(on_audit_single_clicked)

display(Markdown("### 🎛️ Interactive Multi-Class Vulnerability Auditor"))
display(widgets.HBox([suite_selector, audit_single_btn]))
display(suite_output)


### 🎛️ Interactive Multi-Class Vulnerability Auditor

Output()

### 7.1 Automated Batch Suite Audit & Compliance Matrix

Triggering the batch audit executes automated triage across all four vulnerability classes, compiling the results into a centralized **DevSecOps Compliance Matrix**.

In [9]:
batch_audit_btn = widgets.Button(
    description="Run Full Enterprise Batch Audit",
    button_style="success",
    icon="tasks",
    layout=widgets.Layout(width="280px")
)

batch_output = widgets.Output()

def on_batch_audit_clicked(b):
    with batch_output:
        clear_output()
        target_model = model_dropdown.value if "model_dropdown" in globals() else "gemini-3.8-flash"
        display(Markdown(f"### 📊 Running Enterprise Batch Audit across 4 Weakness Classes using `{target_model}`..."))
        
        results = []
        for flaw_name, snippet in VULNERABILITY_SUITE.items():
            display(Markdown(f"*Triage in progress for:* **{flaw_name}**..."))
            prompt = f"""Analyze the code snippet for security vulnerabilities.
Determine the CWE, calculate CVSS score and vector, identify root cause, and synthesize a secure patch.

Code:
{snippet}
"""
            try:
                resp = run_vulnerability_audit(target_model, prompt)
                parsed = DevSecOpsRemediationReport.model_validate_json(resp.text)
                results.append((flaw_name, parsed))
            except Exception as e:
                display(Markdown(f"⚠️ Failed on {flaw_name}: `{e}`"))
        
        table = ["| Vulnerability Class | MITRE CWE | CVSS Score | CVSS Vector | Root Cause Summary |", "|:---|:---|:---|:---|:---|"]
        for name, r in results:
            cwe_link = f"[{r.cwe_id}](https://cwe.mitre.org/data/definitions/{r.cwe_id.replace('CWE-', '')}.html)"
            table.append(f"| **{name}** | {cwe_link} | **{r.cvss_score}** | `{r.cvss_vector}` | {r.root_cause_analysis[:75]}... |")
        
        display(Markdown("### 📋 Executive DevSecOps Compliance Matrix"))
        display(Markdown("\n".join(table)))

batch_audit_btn.on_click(on_batch_audit_clicked)

display(batch_audit_btn)
display(batch_output)


Button(button_style='success', description='Run Full Enterprise Batch Audit', icon='tasks', layout=Layout(widt…

Output()

## 8. Systems Security Engineering Principles & Technical Portfolio Takeaways

### Key Principles for Production Engineering
1. **Language-Level Grammar Invariants:** SQL injection is not an input sanitization failure; it is a parser grammar invariant violation. Never attempt regex-based character blacklisting (`replace("'", "")`). Parameter binding (`?`) is the mathematically sound solution because parameters bypass the lexer and parser entirely.
2. **AST Validation Before Dynamic Compilation:** In autonomous remediation pipelines, always run static compilation (`ast.parse()`) prior to runtime execution to prevent introducing syntax errors into production codebases.
3. **Contract-Driven Structured Outputs:** Using Pydantic v2 schemas (`response_schema`) turns non-deterministic LLM text generation into type-safe, machine-readable CI/CD artifacts.
4. **Closed-Loop Verification:** Automated remediation is incomplete without dynamic validation. Every synthesized patch must undergo exploit neutralization testing and regression testing.

### Technical Interview & Portfolio Framing
> 💡 **How to talk about this project in Application Security / DevSecOps interviews:**
> * "I engineered an autonomous remediation harness that bridges static LLM reasoning with dynamic closed-loop verification."
> * "Instead of relying on fragile prompt parsing, I enforced strict Pydantic contract validation mapped to MITRE CWE taxonomies and CVSS v3.1 scoring."
> * "The system dynamically verifies patches inside an isolated SQLite test sandbox, asserting both exploit neutralization and functional regression prevention before code is admitted to production."
